# 대학원 AI 논문 연구 도우미 v2.2 — Jupyter 실습과 별도 assert 검증

이 노트북은 **인터넷, 공공데이터 API 키, OpenAI API 키 없이** 프로그램의 핵심 로직을 연습합니다. 실제 논문이 아닌 **검증용 가상 PDF**를 생성합니다. 실행 위치는 `app.py`, `pdf_analysis.py`, `prompt_templates.py`, `research_ai.py`가 있는 프로젝트 폴더입니다.

**항상 실습 셀을 실행한 다음 바로 아래 검증 셀을 실행하세요.**

## 실습 1. 105페이지 가상 PDF를 만들고 전체 페이지를 읽기

In [ ]:
# 텍스트를 직접 작성한 예제 PDF (실제 논문 아님)
import fitz
from pdf_analysis import extract_pdf_pages, build_evidence_packets

pdf = fitz.open()
for page_no in range(1, 106):
    page = pdf.new_page()
    heading = "RESULTS" if page_no == 73 else "CONCLUSION" if page_no == 102 else "INTRODUCTION"
    page.insert_text((50, 70), f"{heading}: Page {page_no}: This is a synthetic research document.", fontsize=11)

pdf_bytes = pdf.tobytes()
pdf.close()
source_pdf = extract_pdf_pages(pdf_bytes)
print("전체 페이지:", source_pdf.total_pages)
print("텍스트가 추출된 페이지:", source_pdf.text_pages)
print("마지막 페이지:", source_pdf.pages[-1].text[:90])

In [ ]:
# 검증 셀 1 (별도 실행)
assert source_pdf.total_pages == 105
assert source_pdf.text_pages == 105
assert "Page 105" in source_pdf.pages[104].text
assert "Page 73" in source_pdf.pages[72].text
print("검증 1 통과: 앞 30페이지만 읽는 문제 해결")

## 실습 2. 4개 구간에서 근거 페이지를 골라보기

In [ ]:
packets = build_evidence_packets(source_pdf)
for packet in packets:
    print(packet.section, "→ 페이지", list(packet.page_numbers))

In [ ]:
# 검증 셀 2
selected = {n for packet in packets for n in packet.page_numbers}
assert len(packets) == 4
assert 73 in selected
assert 102 in selected
print("검증 2 통과: 73페이지 결과와 102페이지 결론이 후보에 포함됨")

## 실습 3. 6항목 / 7문장 프롬프트 출력 항목 확인

In [ ]:
from prompt_templates import ASSIGNMENT, READING, fields_for
print("6항목:", fields_for(ASSIGNMENT))
print("7문장:", fields_for(READING))

In [ ]:
# 검증 셀 3
assert len(fields_for(ASSIGNMENT)) == 6
assert len(fields_for(READING)) == 7
assert "주요연구결과" in fields_for(READING)
print("검증 3 통과")

## 실습 4. OpenAI 호출 없이 가짜 응답으로 두 요약 모드 검증

In [ ]:
from research_ai import analyze_paper, analysis_lines

# 가짜 AI 응답을 주는 함수: 실제 API를 호출하지 않습니다.
def fake_ai(system, user, max_tokens=1500):
    if "주요연구결과" in system:
        keys = fields_for(READING)
    else:
        keys = fields_for(ASSIGNMENT)
    return {"items": [{"key": k, "text": "예제 답변이며 연구 사실이 아닙니다.", "evidence": []} for k in keys]}

abstract = "실제 논문이 아닌 연습용으로 데이터와 분석 방법을 설명하는 가상의 초록입니다." * 3
six_result = analyze_paper("", "가상 논문", ASSIGNMENT, abstract=abstract, ask_json=fake_ai)
seven_result = analyze_paper("", "가상 논문", READING, abstract=abstract, ask_json=fake_ai)
print(analysis_lines(seven_result))

In [ ]:
# 검증 셀 4
assert len(six_result["items"]) == 6
assert len(seven_result["items"]) == 7
assert six_result["ai_calls"] == 1
assert seven_result["ai_calls"] == 1
print("검증 4 통과: 두 방식은 독립적으로 생성됩니다")

## 실습 5. 잘못된 PDF 페이지 인용 거부하기

In [ ]:
from pdf_analysis import verify_quote
quote = "RESULTS: Page 73: This is a synthetic research document."
print("73쪽 검증:", verify_quote(source_pdf, 73, quote))
print("74쪽 검증:", verify_quote(source_pdf, 74, quote))

In [ ]:
# 검증 셀 5
assert verify_quote(source_pdf, 73, quote)[0] is True
assert verify_quote(source_pdf, 74, quote)[0] is False
print("검증 5 통과: 다른 페이지의 문구를 근거로 잘못 표시하지 않습니다")

## 실습 6. 읽기용 Excel 보고서·연구노트 확인

In [ ]:
import pandas as pd
from io import BytesIO
from openpyxl import load_workbook
from reporting import create_excel
from research_ai import analysis_summary

paper = {"자료명/저자사항": "가상의 디지털 포렌식 연구", "발행년": "2026", "DB": "사용자 업로드", "controlno": ""}
dataframe = pd.DataFrame([paper])
my_note = "원문을 읽으면서 직접 적은 질문: 다른 데이터베이스에도 적용 가능할까?"
xlsx_bytes = create_excel(dataframe, "가상 포렌식", analysis_summary(seven_result), paper,
                          summary_mode=READING, analysis_result=seven_result, research_note=my_note)
workbook = load_workbook(BytesIO(xlsx_bytes))
print("생성 시트:", workbook.sheetnames)

In [ ]:
# 검증 셀 6
assert "나의_연구노트" in workbook.sheetnames
assert "원문근거_대조" in workbook.sheetnames
assert "AI분석_범위" in workbook.sheetnames
assert my_note in str(workbook["나의_연구노트"].cell(2, 2).value)
print("검증 6 통과: 7문장 연구노트와 분석 정보가 Excel에 포함")

### 추가 검증

전체 프로젝트의 단위 테스트는 Anaconda Prompt에서 `python -m pytest tests -q`로 실행할 수 있습니다. OpenAI 키는 웹앱의 왼쪽 사이드바에만 입력하고, 실습 코드는 `.env`의 실제 인증키를 출력하지 않습니다.
